# FATO_INTERNACAO

Fato de internações hospitalares. **Grão:** 1 registro de AIH paga (arquivo RD do SIH).

- **Chaves:** surrogate keys obtidas por lookup do código natural em cada dimensão;
  código sem correspondência vai para o membro desconhecido (`-1`). Nenhuma linha é descartada.
- **Dimensões degeneradas:** `NUM_AIH`, `TIPO_AIH`, `SEQ_AIH5`, `ANO_MES_CMPT`.
- **PK:** estabelecimento + AIH + tipo + sequencial + competência + data de saída.
- **Medidas:**
  - `QTD_INTERNACAO` (SUM): 1 para AIH normal e 0 para AIH de longa permanência
    (`TIPO_AIH = 5`), que é continuação de uma internação já contada (regra do TabNet);
  - `QTD_DIAS_PERMANENCIA` (SUM; média = soma ÷ internações), `VL_TOTAL` (SUM), `QTD_OBITO` (SUM).

**Performance:** as FKs são desabilitadas pela DAG durante a carga, o que permite
insert direct-path, e reabilitadas com `VALIDATE` depois.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar, coletar_estatisticas

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO FATO_INTERNACAO
    (SK_TEMPO_INTERNACAO, SK_TEMPO_SAIDA, SK_MUNICIPIO_RESIDENCIA, SK_MUNICIPIO_ATENDIMENTO,
     SK_ESTABELECIMENTO, SK_PROCEDIMENTO, SK_CID, SK_ESPECIALIDADE, SK_CARATER, SK_PERFIL,
     NUM_AIH, TIPO_AIH, SEQ_AIH5, ANO_MES_CMPT,
     QTD_INTERNACAO, QTD_DIAS_PERMANENCIA, VL_TOTAL, QTD_OBITO)
SELECT NVL(ti.SK_TEMPO, -1),
       NVL(ts.SK_TEMPO, -1),
       NVL(mr.SK_MUNICIPIO, -1),
       NVL(ma.SK_MUNICIPIO, -1),
       NVL(e.SK_ESTABELECIMENTO, -1),
       NVL(p.SK_PROCEDIMENTO, -1),
       NVL(c.SK_CID, -1),
       NVL(es.SK_ESPECIALIDADE, -1),
       NVL(ca.SK_CARATER, -1),
       NVL(pf.SK_PERFIL, -1),
       s.NUM_AIH,
       s.TIPO_AIH,
       s.SEQ_AIH5,
       s.ANO_MES_CMPT,
       CASE WHEN s.TIPO_AIH = '5' THEN 0 ELSE 1 END,
       s.QTD_DIAS_PERMANENCIA,
       s.VL_TOTAL,
       CASE WHEN s.FLG_OBITO = 'S' THEN 1 ELSE 0 END
FROM SILVER_SIH s
LEFT JOIN DIM_TEMPO ti               ON ti.DT_COMPLETA = s.DT_INTERNACAO
LEFT JOIN DIM_TEMPO ts               ON ts.DT_COMPLETA = s.DT_SAIDA
LEFT JOIN DIM_MUNICIPIO mr           ON mr.COD_IBGE_6 = s.COD_IBGE_6_RES
LEFT JOIN DIM_MUNICIPIO ma           ON ma.COD_IBGE_6 = s.COD_IBGE_6_ATEND
LEFT JOIN DIM_ESTABELECIMENTO e      ON e.COD_CNES = s.COD_CNES
LEFT JOIN DIM_PROCEDIMENTO p         ON p.COD_PROCEDIMENTO = s.COD_PROCEDIMENTO
LEFT JOIN DIM_CID c                  ON c.COD_CID = s.COD_CID
LEFT JOIN DIM_ESPECIALIDADE es       ON es.COD_ESPECIALIDADE = s.COD_ESPECIALIDADE
LEFT JOIN DIM_CARATER_ATENDIMENTO ca ON ca.COD_CARATER = s.COD_CARATER
LEFT JOIN DIM_PERFIL_PACIENTE pf     ON pf.SEXO = s.SEXO AND pf.FAIXA_ETARIA = s.FAIXA_ETARIA
"""

In [ ]:
recarregar("FATO_INTERNACAO", SQL, origem="SILVER_SIH")
coletar_estatisticas("FATO_INTERNACAO")

In [ ]:
consultar("SELECT * FROM FATO_INTERNACAO FETCH FIRST 10 ROWS ONLY")